![](https://i.pinimg.com/originals/51/ae/4b/51ae4b928ea815934b562d907ec5dd91.gif)

<div class="alert alert-block alert-info" style="font-size:20px; background-color: #f3fc9a; font-family:verdana; color: #252908; border: 3px #252908 solid">
    <b>⚕️ MaskRCNN from scratch | Medical Segmentation</b>
    <br>Hi friends! 😉<br>
    Today we will dive into the problem of segmentation of instances in a medical context, and we will also use the MaskRCNN model. I already have work on the topic of instance segmentation, but I believe that a real expert should be able to use not only ready-made pipelines like ultralytics, but also be able to build a pipeline, design a model and train it. It is for this purpose that we are gathered here. Many parts with analysis or explanation of data and tasks are taken from previous work, since I didn’t want to reinvent the wheel (but I changed the gif with bears 🐼) I hope you can learn something new 🙃
</div>

![](https://66.media.tumblr.com/dcae971a5a6e5de34e084e3cf744178e/tumblr_omhrqxDQY91qg39ewo1_500.gif)

<div class="alert alert-block alert-info" style="font-size:20px; background-color: #d1a6ff; font-family:verdana; color: #533078; border: 3px #533078 solid">
    <b>Biological overview ⚕️</b>
    <br>Our task is to segment medical images of kidney tissue samples. Human tissues are covered with many vessels. Our task is to segment such vessels.<br>
</div>

![](https://i.ibb.co/1MZNhNF/003.png)

<div class="alert alert-block alert-info" style="font-size:20px; background-color: #d1a6ff; font-family:verdana; color: #533078; border: 2px #533078 solid">
    <b>Tissue samples 🧪</b>
    <br>Renal cortex: the renal cortex is the outer portion of the kidney that contains round renal corpuscles enclosing glomerular tufts, balls of capillary loops. The renal corpuscle is the start of the nephron, through which the filtration of blood occurs. The renal cortex also contains proximal and distal convoluted tubules, PCTs and DCTs, which are also regions of the nephron. Between these tubular structures is a complex network of capillaries called the peritubular capillaries.<br>
    <br>Renal medulla: the renal medulla is the inner portion of the kidney and is arranged in 8-15 renal pyramids containing linearly arranged tubules comprising the loops of Henle and ducts that gather products for excretion. The capillary network in the renal medulla consists of capillaries called vasa recta. The renal pyramids (medullary tissue) are divided by extensions of the renal cortex called renal columns. There are also projections of the renal medulla into the outer cortex called medullary rays.<br>
    <br>Renal papilla (subsection of medulla): the broad bases of the pyramids connect to the renal cortex at the corticomedullary junctions while the tips form structures called the renal papilla, which project in the minor renal calyces where urine is collected.<br>
</div>

<div class="alert alert-block alert-info" style="font-size:20px; background-color: #b5e1ff; font-family:verdana; color: #011d82; border: 2px #011d82 solid">
    <b>What is instance segmentation?</b>
    <br>Instance Segmentation is a unique form of image segmentation that deals with detecting and delineating each distinct instance of an object appearing in an image. Instance segmentation detects all instances of a class with the extra functionality of demarcating separate instances of any segment class. Hence, it is also referred to as incorporating object detection and semantic segmentation functionality. Watch video to learn more: <a href=instance segmentation>Instance segmentation | Tutorial</a> <br>
</div>

<div class="alert alert-block alert-info" style="font-size:20px; background-color: #b5e1ff; font-family:verdana; color: #011d82; border: 2px #011d82 solid">
    <b>Applications of Semantic Segmentation</b>
<ol><li><strong>Medical Diagnostics: </strong>For detecting medical abnormalities in <a href="https://universe.roboflow.com/search?q=xray&amp;ref=blog.roboflow.com">X-Rays, CT Scans, MRI Scans</a></li><li><strong>GeoSensing:</strong> For land usage mapping from <a href="https://roboflow.com/solutions/aerial?ref=blog.roboflow.com">satellite imagery</a> and monitoring areas of deforestation and urbanization</li><li><strong>Autonomous Driving:</strong> For accurately <a href="https://universe.roboflow.com/browse/self-driving?ref=blog.roboflow.com">detecting lanes, pedestrians, traffic signs, road</a>, sky and other vehicles on the road</li></ol><br>
</div>



<div class="alert alert-block alert-info" style="font-size:20px; background-color: #bdc9bf; font-family:verdana; color: #2b2b2b; border: 2px #2b2b2b solid">
    <b>For determinism 🤗 It's a joke, but we still need reproducibility of experiments</b>
</div>

In [1]:
import numpy as np
import random
import torch
import os

In [2]:
def seed_everythig(seed):
    np.random.seed(seed)
    random.seed(seed)
    os.environ["GLOBALSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True

In [3]:
seed_everythig(42)

# Prepare data
<div class="alert alert-block alert-info" style="font-size:20px; background-color: #abffd1; font-family:verdana; color: #003819; border: 2px #003819 solid">
    <b>It's time to think about data 📚</b>
    <br>For training, we need a dataset, and not just any, but in the COCO format. COCO is an open database for object detection, it is huge and therefore it is used all over the world to evaluate new models according to various metrics. It plays a rather important role, and most detection models are pre-trained on it. That is why this dataset format is used in ultralytics. You don't need to worry about this, as I have already prepared a class for data processing. I advise you to skip the cell below and immediately proceed to the next chapter.<br>
</div>

<div class="alert alert-block alert-info" style="font-size:20px; background-color: #abffd1; font-family:verdana; color: #003819; border: 2px #003819 solid">
    <b>Libraries for creating dataset</b>
</div>

In [4]:
from itertools import chain
import json
import os
import shutil
from tqdm.notebook import tqdm
from colorama import Fore
import yaml

<div class="alert alert-block alert-info" style="font-size:20px; background-color: #abffd1; font-family:verdana; color: #003819; border: 2px #003819 solid">
    <b>Class for creating dataset</b>
</div>

In [5]:
class COCODataset:
    def __init__(self, images_dirpath: str, annotations_filepath: str, length: int = 1633):
        self.train_size = None
        self.val_size = None
        self.length = length
        self.classes = None
        self.labels_counter = None
        self.normalize = None
        
        self.images_dirpath = images_dirpath
        self.annotations_filepath = annotations_filepath
        self.dataset_dirpath = os.path.join(os.getcwd(), "dataset")
        self.train_dirpath =  os.path.join(self.dataset_dirpath, "train")
        self.val_dirpath =  os.path.join(self.dataset_dirpath, "val")
        self.config_path = os.path.join(self.dataset_dirpath, "coco.yaml")

        self.samples = self.parse_jsonl(annotations_filepath)
        self.classes_dict = {
            "background": 0,
            "blood_vessel": 1,
            "glomerulus": 2,
            "unsure": 3,
        }

    def __prepare_dirs(self) -> None:
        if not os.path.exists(self.dataset_dirpath):
            os.makedirs(os.path.join(self.train_dirpath, "images"), exist_ok=True)
            os.makedirs(os.path.join(self.train_dirpath, "labels"), exist_ok=True)
            os.makedirs(os.path.join(self.val_dirpath, "images"), exist_ok=True)
            os.makedirs(os.path.join(self.val_dirpath, "labels"), exist_ok=True)
        else:
            raise RuntimeError("Dataset already exists!")

    def __define_splitratio(self) -> None:
        self.train_size = round(self.length * self.train_size)
        self.val_size = self.length - self.train_size
        assert self.train_size + self.val_size == self.length

    def parse_jsonl(self, path: str) -> list[dict, ...]:
        with open(path, 'r') as json_file:
            jsonl_samples = [
                json.loads(line)
                for line in tqdm(
                    json_file, desc="Processing polygons", total=self.length
                )
            ]
        return jsonl_samples

    def __define_paths(self, i: int) -> dict:
        data_path = self.val_dirpath
        if i < self.train_size:
            data_path = self.train_dirpath
        return {
            "images": os.path.join(data_path, "images"),
            "labels": os.path.join(data_path, "labels")
        }

    @staticmethod
    def __get_label_path(paths_dict: dict, identifier: str) -> str:
        return os.path.join(
            paths_dict["labels"],
            f"{identifier}.txt"
        )

    @staticmethod
    def __get_image_path(paths_dict: dict, identifier: str) -> str:
        return os.path.join(
            paths_dict["images"],
            f"{identifier}.tif"
        )

    def __copy_image(self, dst_path: str, identifier: str) -> str:
        shutil.copyfile(
            os.path.join(self.images_dirpath, f"{identifier}.tif"),
            dst_path
        )

    def __copy_label(self, annotations: list, dst_path: str) -> None:
        with open(dst_path, "w") as file:
            for annotation in annotations:
                coordinates = annotation["coordinates"][0]
                label = self.classes_dict[annotation["type"]]
                if label in self.classes:
                    if coordinates:
                        if self.normalize:
                            coordinates = np.array(coordinates) / 512.0
                        coordinates = " ".join(map(str, chain(*coordinates)))
                        file.write(f"{label} {coordinates}\n")
                        self.labels_counter += 1

    def __splitfolders(self):
        for i, line in tqdm(
                enumerate(self.samples),
                desc="Dataset creation", total=self.length
        ):
            self.labels_counter = 0
            identifier = line["id"]
            annotations = line["annotations"]
            paths_dict = self.__define_paths(i)

            dst_image_path = self.__get_image_path(paths_dict, identifier)
            dst_label_path = self.__get_label_path(paths_dict, identifier)

            self.__copy_image(dst_image_path, identifier)
            self.__copy_label(annotations, dst_label_path)

            if self.labels_counter == 0:
                os.remove(dst_image_path)
                os.remove(dst_label_path)

    def __count_dataset(self) -> dict:
        train_images = len(os.listdir(os.path.join(self.train_dirpath, "images")))
        train_labels = len(os.listdir(os.path.join(self.train_dirpath, "labels")))
        val_images = len(os.listdir(os.path.join(self.val_dirpath, "images")))
        val_labels = len(os.listdir(os.path.join(self.val_dirpath, "labels")))
        return {
            "train_images": train_images,
            "train_labels": train_labels,
            "val_images": val_images,
            "val_labels": val_labels
        }

    @staticmethod
    def __check_sanity(count_dict: dict) -> None:
        assert count_dict["train_images"] == count_dict["train_labels"]
        assert count_dict["val_images"] == count_dict["val_labels"]

    def __finalizing(self, count_dict: dict) -> None:
        assert os.path.exists(self.dataset_dirpath)

        example_structure = [
            "dataset",
            "train", "labels", "images",
            "val", "labels", "images"
        ]

        dir_bone = (
            dirname.split("/")[-1]
            for dirname, _, filenames in os.walk(self.dataset_dirpath)
            if dirname.split("/")[-1] in example_structure
        )

        try:
            print("\n~ HuBMAP Dataset Structure ~\n")
            print(
            f"""
          ├── {next(dir_bone)}
          │   │
          │   ├── {next(dir_bone)}
          │   │   └── {next(dir_bone)}
          │   │   └── {next(dir_bone)}
          │   │
          │   ├── {next(dir_bone)}
          │   │   └── {next(dir_bone)}
          │   │   └── {next(dir_bone)}
            """
            )
        except StopIteration as e:
            print(e)
        else:
            print(Fore.GREEN + "-> Success")
            print(Fore.GREEN + f"Train dataset: {count_dict['train_images']}\nVal dataset: {count_dict['val_images']}")

    def get_config(self) ->dict:
        names = ["background", "blood_vessel", "glomerulus", "unsure"]
        return {
            "train": str(self.train_dirpath),
            "val": str(self.val_dirpath),
            "names": [names[i] for i in self.classes]
        }

    @staticmethod
    def display_config(config: dict) -> None:
        print(Fore.BLACK + "\n~ HuBMAP Config Structure ~\n")
        print(
        f"""
      │   │
      │   ├── train
      │   │   └── {config['train']}/images
      │   │
      │   │
      │   ├── val
      │   │   └── {config['val']}/images
      │   │
      │   │
      │   ├── names
      │   │   └── {' '.join(config['names'])}
        """
        )
        print(Fore.GREEN + "-> Success")
        print(Fore.GREEN + f"Number of classes: {len(config['names'])}"
                           f"\nClasses: {' '.join(config['names'])}" 
              )

    def write_config(self, config: dict) -> None:
        with open(self.config_path, mode="w") as f:
            yaml.safe_dump(stream=f, data=config)

    def __call__(self, train_size: float,
                 classes: list[int, ...],
                 make_config: bool = True,
                 normalize: bool = True
                ) -> None:
        
        self.train_size = train_size
        self.classes = classes
        self.normalize = normalize
        
        self.__define_splitratio()
        self.__prepare_dirs()
        self.__splitfolders()
        count_dict = self.__count_dataset()
        self.__check_sanity(count_dict)
        self.__finalizing(count_dict)
        
        if make_config:
            config = self.get_config()
            self.write_config(config)
            self.display_config(config)

<div class="alert alert-block alert-info" style="font-size:20px; background-color: #abffd1; font-family:verdana; color: #003819; border: 2px #003819 solid">
    <b>Let's fire up the data 🔥</b>
</div>

In [6]:
coco = COCODataset(
    annotations_filepath="/kaggle/input/hubmap-hacking-the-human-vasculature/polygons.jsonl",
    images_dirpath="/kaggle/input/hubmap-hacking-the-human-vasculature/train",
) 

Processing polygons:   0%|          | 0/1633 [00:00<?, ?it/s]

In [7]:
coco(train_size=0.85, classes=[1, 2, 3], normalize=False)

Dataset creation:   0%|          | 0/1633 [00:00<?, ?it/s]


~ HuBMAP Dataset Structure ~


          ├── dataset
          │   │
          │   ├── val
          │   │   └── labels
          │   │   └── images
          │   │
          │   ├── train
          │   │   └── labels
          │   │   └── images
            
-> Success
Train dataset: 1388
Val dataset: 245

~ HuBMAP Config Structure ~


      │   │
      │   ├── train
      │   │   └── /kaggle/working/dataset/train/images
      │   │
      │   │
      │   ├── val
      │   │   └── /kaggle/working/dataset/val/images
      │   │
      │   │
      │   ├── names
      │   │   └── blood_vessel glomerulus unsure
        
-> Success
Number of classes: 3
Classes: blood_vessel glomerulus unsure


# About Mask-RCNN
<div class="alert alert-block alert-info" style="font-size:20px; background-color: #a1e6c4; font-family:verdana; color: #004725; border: 2px #004725 solid">
    <b>Understanding Architecture</b>
    <br>Here I would like to share information about the Mask-RCNN architecture and its development<br>
</div>

![](https://media.tenor.com/J5zw2jZ6TsYAAAAC/ice-bear-math-lady.gif)

<div class="alert alert-block alert-info" style="font-size:20px; background-color: #a1e6c4; font-family:verdana; color: #004725; border: 2px #004725 solid">
    <b>R-CNN (Regions With CNNs)</b>
    <br>The R-CNN (Regions With CNNs) network architecture was developed by a team at UC Berkley to apply Convolution Neural Networks to the object detection problem. The approaches to solving such problems that existed at that time approached the maximum of their capabilities and it was not possible to significantly improve their performance.<br>
    <br>CNNs have been good at image classification, and in this network they have essentially been applied to do the same thing. To do this, not the entire image was fed to the input of the CNN, but regions previously selected in a different way, in which presumably there are some objects. At that time there were several such approaches, the authors chose Selective Search, although they indicate that there are no special reasons for preferring it.<br>
    <br>A ready-made architecture, CaffeNet (AlexNet), was also used as a CNN network. Such neural networks, like others for the ImageNet image set, classify into 1000 classes. R-CNN was designed to detect objects of fewer classes (N=20 or 200), so the last CaffeNet classification layer was replaced with a layer with N+1 outputs (with an additional class for the background).<br>
    <br>Selective Search returned about 2000 regions of different sizes and aspect ratios, but CaffeNet accepts images of a fixed size of 227x227 pixels as input, so they had to be modified before submitting the regions to the network input. To do this, the image from the region was enclosed in the smallest enclosing square. Along that (smaller) side on which the fields were formed, several “context” (surrounding the region) pixels of the image were added, the rest of the field was not filled with anything. The resulting square was scaled to 227x227 and fed to the CaffeNet input.<br>
    <br>Even though the CNN was trained to recognize N+1 classes, it ended up being used only to extract a fixed 4096-dimensional feature vector. N linear SVMs were engaged in the direct determination of the object in the image, each of which carried out a binary classification according to its type of objects, determining whether there is one in the transferred region or not. In the original document, the whole procedure is illustrated by the following diagram:<br>
</div>

![](https://habrastorage.org/r/w1560/webt/6i/zz/rh/6izzrhggbprdgbpbqz-qa7lfreu.jpeg)

<div class="alert alert-block alert-info" style="font-size:20px; background-color: #a1e6c4; font-family:verdana; color: #004725; border: 2px #004725 solid">
    <b>Fast R-CNN</b>
    <br>Despite the good results, the performance of R-CNN was still poor, especially for networks deeper than CaffeNet (such as VGG16). In addition, training the bounding box regressor and SVM required a large number of features to be saved to disk, so it was expensive in terms of storage size. The authors of Fast R-CNN suggested speeding up the process with a couple of modifications:<br>
    <ul>
        <li>Pass through CNN not each of the 2000 candidate regions separately, but the entire image. The proposed regions are then superimposed on the resulting overall feature map;</li>
        <li>Instead of training three models (CNN, SVM, bbox regressor) independently, combine all training procedures into one.</li>
    </ul>
    <br>The transformation of features that fell into different regions to a fixed size was performed using the RoIPooling procedure. The region window of width w and height h was divided into a grid with H × W cells of size h/H × w/W. (Document authors used W=H=7). For each such cell, Max Pooling was performed to select only one value, thus giving the resulting H×W feature matrix.<br>
    <br>Binary SVMs were not used, instead the selected features were passed to a fully connected layer and then to two parallel layers: softmax with K+1 outputs (one for each class + 1 for the background) and a bounding box regressor. The general network architecture looks like this:<br>
</div>

![](https://habrastorage.org/r/w1560/webt/yt/e-/9u/yte-9u2kp27hykwg9m1n98w6upg.png)

<div class="alert alert-block alert-info" style="font-size:20px; background-color: #a1e6c4; font-family:verdana; color: #004725; border: 2px #004725 solid">
    <b>Faster R-CNN</b>
    <br>After the improvements made in Fast R-CNN, the mechanism for generating candidate regions turned out to be the bottleneck of the neural network. In 2015, the team at Microsoft Research was able to make this phase much faster. They proposed to calculate the regions not from the original image, but again from the feature map obtained from CNN. To do this, a module called the Region Proposal Network (RPN) was added. The new architecture looks like this:<br>
</div>

![](https://habrastorage.org/r/w1560/webt/jf/-3/22/jf-3224hkrudwxi1_7oogb1avg0.png)

<div class="alert alert-block alert-info" style="font-size:20px; background-color: #a1e6c4; font-family:verdana; color: #004725; border: 2px #004725 solid">
Within the framework of RPN, a “mini-neural network” with a small (3x3) window slides along the extracted CNN features. The values obtained with its help are transferred to two parallel fully connected layers: box-regression layer (reg) and box-classification layer (cls). The outputs of these layers are based on the so-called anchors: k frames for each position of the sliding window, with different sizes and aspect ratios. The reg layer for each such anchor produces 4 coordinates that adjust the position of the enclosing frame; cls-layer produces two numbers each - the probabilities that the frame contains at least some object or that it does not. This is illustrated in the document as follows:
</div>

![](https://habrastorage.org/r/w1560/webt/kj/kj/oz/kjkjozq2gq70fuvf77dbml8-_nw.png)

<div class="alert alert-block alert-info" style="font-size:20px; background-color: #a1e6c4; font-family:verdana; color: #004725; border: 2px #004725 solid">
    <b>Mask R-CNN</b>
<br>Mask R-CNN develops the Faster R-CNN architecture by adding another branch that predicts the position of the mask covering the found object, and thus solves the instance segmentation problem. The mask is simply a rectangular matrix, in which 1 at some position means that the corresponding pixel belongs to an object of a given class, 0 means that the pixel does not belong to the object.<br>
</div>

![](https://habrastorage.org/r/w1560/webt/n3/cs/tp/n3cstpty6ktfwhw6vklswah1rxk.png)

<div class="alert alert-block alert-info" style="font-size:20px; background-color: #a1e6c4; font-family:verdana; color: #004725; border: 2px #004725 solid">
    <b>Do you feel? That feeling of realizing what you're using 🙃</b>
<br>This is a squeeze that, in principle, should give you an idea of ​​​​the history of the development of Mask-RCNN and its architecture. I know I didn’t mention the intricacies of training and composite loss functions, but I decided to leave this to your interest so as not to drag out the notebook<br>
</div>

In [8]:
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor

In [9]:
def get_model(num_classes: int):
    # load an instance segmentation model pre-trained on COCO
    model = torchvision.models.detection.maskrcnn_resnet50_fpn(weights="DEFAULT")

    # get number of input features for the classifier
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    # replace the pre-trained head with a new one
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

    # now get the number of input features for the mask classifier
    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    hidden_layer = 256
    # and replace the mask predictor with a new one
    model.roi_heads.mask_predictor = MaskRCNNPredictor(in_features_mask,
                                                       hidden_layer,
                                                       num_classes)

    return model

In [10]:
model = get_model(4)

Downloading: "https://download.pytorch.org/models/maskrcnn_resnet50_fpn_coco-bf2d0c1e.pth" to /root/.cache/torch/hub/checkpoints/maskrcnn_resnet50_fpn_coco-bf2d0c1e.pth
100%|██████████| 170M/170M [00:00<00:00, 239MB/s] 


<div class="alert alert-block alert-info" style="font-size:20px; background-color: #ffd37a; font-family:verdana; color: #543800; border: 2px #543800 solid">
    <b>What about transforming our data?</b>
<br>Firstly, we absolutely need to bring the pictures to a single size and also normalize them. It would also not be bad to augment our data. But here's the problem. After all, we are now working not with an image and its single mask as in semantic segmentation, but with individual instances, their masks and boxes. In this case, we can ignore color transformations, but all spatial transformations change both box and mask coordinates. In this case, we must take this into account. A wonderful library from the community comes to the rescue - albumentations. It will allow you to quickly and easily take into account all the transformations. Otherwise, we had to read it ourselves.<br>
</div>

![](https://www.researchgate.net/publication/319413978/figure/fig2/AS:533727585333249@1504261980375/Data-augmentation-using-semantic-preserving-transformation-for-SBIR.png)

In [11]:
import albumentations as A
from albumentations.pytorch.transforms import ToTensorV2

<div class="alert alert-block alert-info" style="font-size:20px; background-color: #ffd37a; font-family:verdana; color: #543800; border: 2px #543800 solid">
    <b>Transforms for training</b>
</div>

In [12]:
train_transforms = [
    A.Resize(512, 512, p=1), 
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.45),
    A.HueSaturationValue(p=0.35),
    
    A.OneOf([
        A.MotionBlur(),
        A.Blur(blur_limit=3),
        A.MedianBlur(blur_limit=3),
        A.GaussNoise()
        ], p=0.10
    ),
                         
    A.Normalize(),
    ToTensorV2()
]

<div class="alert alert-block alert-info" style="font-size:20px; background-color: #ffd37a; font-family:verdana; color: #543800; border: 2px #543800 solid">
    <b>And for validation</b>
</div>

In [13]:
val_transforms = [
    A.Resize(512, 512, p=1), 
    A.Normalize(),
    ToTensorV2()
]

# Dataset
<div class="alert alert-block alert-info" style="font-size:20px; background-color: #859cd6; font-family:verdana; color: #001342; border: 2px #001342 solid">
    <b>It's time to take care of our dataset</b>
    <br>It should act as an interface for getting images and a dictionary with target data. What I'm talking about?<br>
</div>

<div class="alert alert-block alert-info" style="font-size:20px; background-color: #859cd6; font-family:verdana; color: #001342; border: 2px #001342 solid">
<ul class="simple">
<li><p>image: a PIL Image of size <code class="docutils literal notranslate"><span class="pre">(H,</span> <span class="pre">W)</span></code></p></li>
<li><p>target: a dict containing the following fields</p>
<ul>
<li><p><code class="docutils literal notranslate"><span class="pre">boxes</span> <span class="pre">(FloatTensor[N,</span> <span class="pre">4])</span></code>: the coordinates of the <code class="docutils literal notranslate"><span class="pre">N</span></code>
bounding boxes in <code class="docutils literal notranslate"><span class="pre">[x0,</span> <span class="pre">y0,</span> <span class="pre">x1,</span> <span class="pre">y1]</span></code> format, ranging from <code class="docutils literal notranslate"><span class="pre">0</span></code>
to <code class="docutils literal notranslate"><span class="pre">W</span></code> and <code class="docutils literal notranslate"><span class="pre">0</span></code> to <code class="docutils literal notranslate"><span class="pre">H</span></code></p></li>
<li><p><code class="docutils literal notranslate"><span class="pre">labels</span> <span class="pre">(Int64Tensor[N])</span></code>: the label for each bounding box. <code class="docutils literal notranslate"><span class="pre">0</span></code> represents always the background class.</p></li>
<li><p><code class="docutils literal notranslate"><span class="pre">image_id</span> <span class="pre">(Int64Tensor[1])</span></code>: an image identifier. It should be
unique between all the images in the dataset, and is used during
evaluation</p></li>
<li><p><code class="docutils literal notranslate"><span class="pre">area</span> <span class="pre">(Tensor[N])</span></code>: The area of the bounding box. This is used
during evaluation with the COCO metric, to separate the metric
scores between small, medium and large boxes.</p></li>
<li><p><code class="docutils literal notranslate"><span class="pre">iscrowd</span> <span class="pre">(UInt8Tensor[N])</span></code>: instances with iscrowd=True will be
ignored during evaluation.</p></li>
<li><p>(optionally) <code class="docutils literal notranslate"><span class="pre">masks</span> <span class="pre">(UInt8Tensor[N,</span> <span class="pre">H,</span> <span class="pre">W])</span></code>: The segmentation
masks for each one of the objects</p></li>
<li><p>(optionally) <code class="docutils literal notranslate"><span class="pre">keypoints</span> <span class="pre">(FloatTensor[N,</span> <span class="pre">K,</span> <span class="pre">3])</span></code>: For each one of
the N objects, it contains the K keypoints in
<code class="docutils literal notranslate"><span class="pre">[x,</span> <span class="pre">y,</span> <span class="pre">visibility]</span></code> format, defining the object. visibility=0
means that the keypoint is not visible. Note that for data
augmentation, the notion of flipping a keypoint is dependent on
the data representation, and you should probably adapt
<code class="docutils literal notranslate"><span class="pre">references/detection/transforms.py</span></code> for your new keypoint
representation</p></li>
</ul>
</li>
</ul>
</div>

In [14]:
from torch.utils.data import Dataset
import albumentations as A
import torchvision.transforms as T
import cv2
import os
import yaml
from typing import Literal, Any, Union
from tqdm import tqdm
import json

In [15]:
class HuBMAPDataset(Dataset):
    def __init__(self,
                 stage: Literal["train", "val"],
                 config_path: str,
                 transforms: Union[A.Compose, T.Compose] = None,
                 *args, **kwargs
                 ):
        
        self.config = self.load_config(config_path)
        self.images_dirpath = None
        self.labels_dirpath = None
        self.__define_paths(stage)
        
        self.names = self.config["names"]
        self.num_classes = len(self.names) 
        self.samples = os.listdir(self.labels_dirpath)
        
        if transforms:
            self.bbox_params = {
                "format":"pascal_voc",
                "min_area": 0,
                "min_visibility": 0,
                "label_fields": ["category_id"]
            }
            self.transforms = A.Compose(transforms, bbox_params=self.bbox_params)
        
    @staticmethod
    def load_config(path: str) -> dict:
        with open(path, mode="r") as f:
            data = yaml.load(stream=f, Loader=yaml.SafeLoader)
        return data

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, dict]:
        filename = self.samples[idx].split(".")[0]
        paths = self._get_paths(filename)
        image = cv2.imread(paths["image"], cv2.COLOR_BGR2RGB)  
        target = self._get_target(paths["label"])
        target["image_id"] = torch.tensor([idx])
        if self.transforms:
            image, target = self.transform(image, target)
        return image, target
    
    def transform(self, image: np.ndarray, target: dict) -> tuple[torch.Tensor, dict]:
        transformed = self.transforms(
            image=image, masks=target["masks"],
            bboxes=target["boxes"], 
            category_id=target["labels"]
        )
    
        image = transformed["image"]
        target["masks"] = torch.as_tensor(
            np.array(list(map(np.array, transformed["masks"])), dtype=np.uint8)
        ) 
        
        target["labels"] = torch.tensor(transformed["category_id"])
        target["boxes"] = torch.as_tensor(transformed["bboxes"], dtype=torch.float32)
        target["area"] = self.__get_area(target["boxes"])
        return image, target
        
    def __define_paths(self, stage: Literal["train", "val"]) -> None:
        data_dirpath = self.config[stage]
        self.images_dirpath = os.path.join(data_dirpath, "images")
        self.labels_dirpath = os.path.join(data_dirpath, "labels")
        
    def _get_paths(self, filename: str) -> dict:
        image_path = os.path.join(self.images_dirpath, f"{filename}.tif")
        label_path = os.path.join(self.labels_dirpath, f"{filename}.txt")
        return {
            "image": image_path,
            "label": label_path
        }
    
    @staticmethod
    def _get_target_sample() -> dict:
        return {
            "boxes": [],
            "masks": [],
            "area": [],
            "labels": [],
            "iscrowd": None,
            "image_id": None
        }

    def _get_target(self, annotations_path: str) -> dict:
        target = self._get_target_sample()

        with open(annotations_path, "r") as file:
            for line in file:
                label = int(line[0])
                coordinates = np.array(list(map(int, line[1:].split()))).reshape(1, -1, 2)
                mask = self.__get_mask(label, coordinates)
                box = self.__get_box(mask)
                target["masks"].append(mask)    
                target["boxes"].append(box)
                target["labels"].append(label)
                
        num_objs = len(target["labels"])
        target["iscrowd"] = torch.zeros((num_objs,), dtype=torch.int64)
        return target 

    @staticmethod
    def __get_mask(label: int, coordinates: np.ndarray) -> np.ndarray:
        mask = np.zeros((512, 512), dtype=np.uint8)
        return cv2.fillPoly(
            mask, pts=coordinates,
            color=(label, label, label)
        )
    
    @staticmethod
    def __get_box(mask: np.ndarray) -> list[np.ndarray, ...]:
        pos = np.nonzero(mask)
        xmin = np.min(pos[1])
        xmax = np.max(pos[1])
        ymin = np.min(pos[0])
        ymax = np.max(pos[0])
        return [xmin, ymin, xmax, ymax]
    
    @staticmethod
    def __get_area(boxes: list[list, ...]) -> torch.Tensor:
         return (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0])

<div class="alert alert-block alert-info" style="font-size:20px; background-color: #859cd6; font-family:verdana; color: #001342; border: 2px #001342 solid">
    <b>Dataset for training</b>
</div>

In [16]:
train_dataset = HuBMAPDataset(
    stage="train",
    config_path=coco.config_path,
    transforms=train_transforms
)

<div class="alert alert-block alert-info" style="font-size:20px; background-color: #859cd6; font-family:verdana; color: #001342; border: 2px #001342 solid">
    <b>Dataset for validation</b>
</div>

In [17]:
val_dataset = HuBMAPDataset(
    stage="val",
    config_path=coco.config_path,
    transforms=val_transforms
)

# Dataloaders
<div class="alert alert-block alert-info" style="font-size:20px; background-color: #b7ebea; font-family:verdana; color: #233d3d; border: 2px #233d3d solid">
    <b>And how to manipulate the unloading of data?</b>
    <br>Imagine that for training the neural network there are mastaba data, which should remain less in the RAM for a short while and stay in the video memory for a longer time. This is quite costly in terms of storage. But what to do? Make a lazy iterator. At least before, the output was like this, as soon as the loader issued the batch, it immediately appeared in memory, and disappeared from the iterator, and at the same time, the iterator itself contained only information about the object, and not the object itself. However, now we do not need to worry about such low-level things and we are engaged in the Dataloader sports class. You, in turn, now know why it is needed.<br>
</div>

In [18]:
from torch.utils.data import DataLoader

<div class="alert alert-block alert-info" style="font-size:20px; background-color: #b7ebea; font-family:verdana; color: #233d3d; border: 2px #233d3d solid">
    <b>Dataloader for training</b>
</div>

In [19]:
train_dataloader = DataLoader(
    dataset=train_dataset,
    batch_size=16, 
    shuffle=True,
    pin_memory=True,
    num_workers=4,
    collate_fn=lambda x: tuple(zip(*x))
)

/opt/conda/lib/python3.10/site-packages/torch/utils/data/dataloader.py:561: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(


<div class="alert alert-block alert-info" style="font-size:20px; background-color: #b7ebea; font-family:verdana; color: #233d3d; border: 2px #233d3d solid">
    <b>And for validation</b>
</div>

In [20]:
val_dataloader = DataLoader(
    dataset=val_dataset,
    batch_size=16, 
    shuffle=False,
    pin_memory=True,
    num_workers=4,
    collate_fn=lambda x: tuple(zip(*x))
)

# Logging
<div class="alert alert-block alert-info" style="font-size:20px; background-color: #eec4ff; font-family:verdana; color: #511b66; border: 2px #511b66 solid">
    <b>And now a little about the control of experiments</b>
    <br>Experiment control tools are a very important part of your pipeline. With the help of them, you can track learning and compare results with previous ones to solve your problem. It is very convenient and allows you not to get confused.<br>
</div>

# Weights and biases
  
 <p align='center'> 
 <a href="https://pypi.python.org/pypi/wandb"><img src="https://img.shields.io/pypi/v/wandb" /></a> 
 <a href="https://anaconda.org/conda-forge/wandb"><img src="https://img.shields.io/conda/vn/conda-forge/wandb" /></a> 
 <a href="https://circleci.com/gh/wandb/wandb"><img src="https://img.shields.io/circleci/build/github/wandb/wandb/main" /></a> 
 <a href="https://codecov.io/gh/wandb/wandb"><img src="https://img.shields.io/codecov/c/gh/wandb/wandb" /></a> 
 </p> 
 <p align='center'> 
 <a href="https://colab.research.google.com/github/wandb/examples/blob/master/colabs/intro/Intro_to_Weights_%26_Biases.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" /></a> 
 </p> 
  
 Use W&B to build better models faster. Track and visualize all the pieces of your machine learning pipeline, from datasets to production machine learning models. Get started with W&B today, [sign up for a free account!](https://wandb.com?utm_source=github&utm_medium=code&utm_campaign=wandb&utm_content=readme) 
  
 🎓 W&B is free for students, educators, and academic researchers. For more information, visit [https://wandb.ai/site/research](https://wandb.ai/site/research?utm_source=github&utm_medium=code&utm_campaign=wandb&utm_content=readme). 
  
 Want to use Weights & Biases for seamless collaboration between your ML or Data Science team? Looking for Production-grade MLOps at scale? Sign up to one of [our plans](https://wandb.ai/site/pricing) or [contact the Sales Team](https://wandb.ai/site/contact).


<div class="alert alert-block alert-info" style="font-size:20px; background-color: #eec4ff; font-family:verdana; color: #511b66; border: 2px #511b66 solid">
    <b>Please follow the <a href=https://wandb.ai/site>link</a>. Sign up and then paste your API key into the box that pops up below. Do not disclose your key to anyone. Here it is required so that you can track the training</b>
</div>

![](https://i.ibb.co/q1xnxnD/2023-07-01-06-42-42.png)

In [21]:
import wandb

wandb.login() 

run = wandb.init(
    # Set the project where this run will be logged
    project="HuBMAP-Mask-RCNN",
    # Track hyperparameters and run metadata
    config={
        "learning_rate": 0.0035,
        "epochs": 15,
    })

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit:

  ········································


wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: entertomerci (umbrella_team). Use `wandb login --relogin` to force relogin


# Training
<div class="alert alert-block alert-info" style="font-size:20px; background-color: #bcf5a9; font-family:verdana; color: #1c3d11; border: 2px #1c3d11 solid">
    <b>Finally, we can start learning</b>
    <br>Yes, yes I know. I was just talking about high-level and how abstraction can be annoying, but look around. We use pure pytorch without wrappers. However, all work with losses is registered under the hood of the model class. The model itself is just as seriously provided to us and everyone can assemble it as a lego constructor. In general, you do not need to be afraid of abstraction, there is no escape from it, but I am against high-level if it allows you to remain in the dark and continue using the tool.<br>
</div>

In [22]:
from tqdm.notebook import tqdm
import torch.nn as nn
from torch import optim

In [23]:
class Trainer:
    def __init__(self,
                 model: nn.Module,
                train_dataloader: DataLoader,
                 val_dataloader: DataLoader,
                 early_stop: dict = {"monitor": "loss_mask", "patience": 5},
                 save_every_epoch: int = 1,
                 save_dirpath: str = "/kaggle/working/runs"
                ):
        
        # Callbacks | Early stoping & Model checkpoint
        self.patience = early_stop["patience"]
        self.monitor = early_stop["monitor"]
        self.track_list = []
        self.save_every_epoch = save_every_epoch
        self.save_dirpath = save_dirpath
        
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.train_dataloader = train_dataloader
        self.val_dataloader = val_dataloader
        self.train_batches = len(train_dataloader)
        self.val_batches =  len(val_dataloader)
        
        self.model = model
        self.setup_model()
        
        self.optim_dict = self.configure_optimizers()
        self.optimizer = self.optim_dict["optimizer"]
        self.lr_scheduler = self.optim_dict["lr_scheduler"]
        
        self.step_outputs = {
            "loss_objectness": 0,
            "loss_mask": 0,
            "loss_classifier": 0,
            "loss_rpn_box_reg": 0,
            "loss_box_reg": 0,
            "loss": 0
        }
        
    def configure_optimizers(self) -> dict:
        # construct an optimizer
        params = [
            p
            for p in self.model.parameters()
            if p.requires_grad
        ]

        optimizer = optim.SGD(
            params,
            lr=0.0018,
            momentum=0.938,
            weight_decay=0.00053
        )

        # and a learning rate scheduler
        lr_scheduler = torch.optim.lr_scheduler.StepLR(
            optimizer,
            step_size=3,
            gamma=0.1
        )
        return {
            "lr_scheduler": lr_scheduler,
            "optimizer": optimizer
        }
    
    def setup_model(self) -> None:
        for param in self.model.parameters():
            param.requires_grad = True
        self.model.to(self.device)
        self.model.train()
    
    def to_device(self, batch: tuple) -> tuple:
        images, targets = batch
        images = list(image.to(self.device) for image in images)

        targets = [
            {key: value.to(self.device) 
             for key, value in target.items()}
            for target in targets
        ]
        
        return images, targets
    
    def training_step(self, batch) -> dict:
        images, targets = self.to_device(batch)
        self.optimizer.zero_grad() 
        outputs = self.model(images, targets)
        loss = sum([loss for loss in outputs.values()])
        outputs["loss"] = loss
        loss.backward()
        self.optimizer.step()
        self.lr_scheduler.step()
        return outputs
    
    def validation_step(self, batch) -> dict:
        images, targets = self.to_device(batch)
        with torch.no_grad():
            outputs = self.model(images, targets)
            loss = sum([loss for loss in outputs.values()])
            outputs["loss"] = loss
        return outputs
    
    def shared_epoch_end(self, stage: str, epoch: int) -> float:
        tracked_loss = self.step_outputs[self.monitor]    
        loss_objectness = self.step_outputs["loss_objectness"]
        loss_mask = self.step_outputs["loss_mask"]
        loss_classifier = self.step_outputs["loss_classifier"]
        loss_rpn_box_reg = self.step_outputs["loss_rpn_box_reg"]
        loss_box_reg = self.step_outputs["loss_box_reg"]
        loss = self.step_outputs["loss"]
        
        wandb.log({
            f"{stage}_loss_objectness": loss_objectness,
            f"{stage}_loss_mask": loss_mask,
            f"{stage}_loss_classifier": loss_classifier,
            f"{stage}_loss_rpn_box_reg": loss_rpn_box_reg,
            f"{stage}_loss_box_reg": loss_box_reg,
            f"{stage}_loss": loss    
        })
        
        print(
            f"""
            || End {epoch} {stage} epoch ||
            loss_objectness: {loss_objectness:.2f}
            loss_mask: {loss_mask:.2f}
            loss_classifier: {loss_classifier:.2f}
            loss_rpn_box_reg: {loss_rpn_box_reg:.2f}
            loss_box_reg: {loss_box_reg:.2f} 
            loss: {loss:.2f}\n
            """
        )
              
        self.step_outputs = self.step_outputs.fromkeys(self.step_outputs, 0)
        if stage == "val":
            return tracked_loss
              
    def on_train_epoch_end(self, epoch: int) -> None:
        return self.shared_epoch_end(stage="train", epoch=epoch)

    def on_validation_epoch_end(self, epoch: int) -> None:
        tracked_loss = self.shared_epoch_end(stage="val", epoch=epoch)
        patience = 0
        
        if epoch > self.patience:
            last_tracked = list(reversed(self.track_list))[:self.patience]
            for i in last_tracked:
                if i <= tracked_loss:
                    patience += 1
                    
        self.track_list.append(tracked_loss)
        return tracked_loss, (self.patience - patience)

    def train(self, max_epochs: int) -> None:
        
        for epoch in range(1, max_epochs + 1):
            for batch_idx, batch in tqdm(enumerate(self.train_dataloader, 1), desc="Training", total=self.train_batches, colour="#068e58"):
                outputs = self.training_step(batch)
                for key, value in outputs.items():
                    self.step_outputs[key] += float(value.detach().cpu().numpy()) / self.train_batches
            self.on_train_epoch_end(epoch)

            for batch_idx, batch in tqdm(enumerate(val_dataloader, 1), desc="Validation", total=self.val_batches, colour="#013385"):
                outputs = self.validation_step(batch)
                for key, value in outputs.items():
                    self.step_outputs[key] += float(value.detach().cpu().numpy()) / self.val_batches
            tracked_loss, patience = self.on_validation_epoch_end(epoch)
            
            if epoch % self.save_every_epoch == 0:
                if not os.path.exists(self.save_dirpath):
                    os.mkdir(self.save_dirpath)
                path = os.path.join(self.save_dirpath, f"epoch_{epoch}_{self.monitor}_{tracked_loss:.2f}.pt")
                torch.save(model.state_dict(), path) 
                print("\nThe model passed the save checkpoint successfully!\n")
                
            if patience == 0:
                print("Our patience has run out! Model training stopped beforehand.")
                break


In [24]:
trainer = Trainer(
    model=model,
    train_dataloader=train_dataloader,
    val_dataloader=val_dataloader,
    early_stop = {"monitor": "loss_mask", "patience": 5},
    save_every_epoch=1
)

In [25]:
trainer.train(max_epochs=15)

/opt/conda/lib/python3.10/site-packages/torch/utils/data/dataloader.py:561: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(


Training:   0%|          | 0/87 [00:00<?, ?it/s]


            || End 1 train epoch ||
            loss_objectness: 0.52
            loss_mask: 0.57
            loss_classifier: 0.37
            loss_rpn_box_reg: 0.10
            loss_box_reg: 0.21 
            loss: 1.77

            


Validation:   0%|          | 0/16 [00:00<?, ?it/s]


            || End 1 val epoch ||
            loss_objectness: 0.42
            loss_mask: 0.53
            loss_classifier: 0.37
            loss_rpn_box_reg: 0.09
            loss_box_reg: 0.23 
            loss: 1.63

            

The model passed the save checkpoint successfully!



...


            || End 7 val epoch ||
            loss_objectness: 0.41
            loss_mask: 0.53
            loss_classifier: 0.37
            loss_rpn_box_reg: 0.09
            loss_box_reg: 0.23 
            loss: 1.62

            

The model passed the save checkpoint successfully!



Training:   0%|          | 0/87 [00:00<?, ?it/s]


            || End 8 train epoch ||
            loss_objectness: 0.49
            loss_mask: 0.53
            loss_classifier: 0.36
            loss_rpn_box_reg: 0.10
            loss_box_reg: 0.21 
            loss: 1.68

            


Validation:   0%|          | 0/16 [00:00<?, ?it/s]


            || End 8 val epoch ||
            loss_objectness: 0.41
            loss_mask: 0.53
            loss_classifier: 0.37
            loss_rpn_box_reg: 0.09
            loss_box_reg: 0.23 
            loss: 1.62

            

The model passed the save checkpoint successfully!

Our patience has run out! Model training stopped beforehand.


<div class="alert alert-block alert-info" style="font-size:20px; background-color: #bcf5a9; font-family:verdana; color: #1c3d11; border: 2px #1c3d11 solid">
    <b>Wrapper class for Mask-RCNN model</b>
    <br>So it will be more convenient to look at the fruits of our labors, as well as make a submit<br>
</div>

In [26]:
class BestMaskRCNN:
    def __init__(self, 
                 model: nn.Module,
                 checkpoint_path: str,
                 conf: float = 0.05
                ):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.checkpoint_path = checkpoint_path
        self.model = self.get_model(model)
        self.conf = conf
        self.transforms = A.Compose([
            A.Resize(512, 512, p=1), 
            A.Normalize(),
            ToTensorV2()
        ])
        
    def to_cpu(self, outputs: dict) -> tuple:
        outputs = {
            key: value.cpu().numpy()
            for key, value in outputs.items()
        }       
        return outputs
    
    def get_model(self, model: nn.Module) -> nn.Module:
        checkpoint = torch.load(self.checkpoint_path) 
        model.load_state_dict(checkpoint) 
        model.eval()
        model.to(self.device)
        for param in model.parameters():
            param.requires_grad = False
        return model
    
    def get_image(self, path: str):
        image = cv2.imread(path, cv2.COLOR_BGR2RGB)  
        return self.transforms(image=image)["image"]
    
    def forward(self, path: str):
        image = self.get_image(path)
        
        image = torch.as_tensor(
            np.expand_dims(image.numpy(), 0),
            dtype=image.dtype
        ).to(self.device)
        
        outputs = self.model(image)[0]
        outputs = self.to_cpu(outputs)
        return outputs
    
    def __call__(self, source: str) -> list[dict, ...]:
        sublist = []
        result = self.forward(source)

        for i in range(len(result["masks"])):
            conf = round(float(result["scores"][i]), 2)
            mask = result["masks"][i].transpose(1,2,0)

            if int(result["labels"][i]) == 1 and conf >= self.conf:
                sublist.append({"mask": mask, "confidence": conf})
            else:
                continue
        return sublist

In [28]:
model = BestMaskRCNN(
    model=get_model(4),
    checkpoint_path="/kaggle/working/runs/epoch_2_loss_mask_0.53.pt"
)

<div class="alert alert-block alert-info" style="font-size:20px; background-color: #f2ffb0; font-family:verdana; color: #3c450e; border: 2px #3c450e solid">
    <b>And how can I use the right tools if the internet is banned in the competition? 🙁</b>
    <br>In the competition, many have difficulty installing the necessary tools and using them, because the Internet is prohibited, but it is prohibited only so that the test data is not stolen (they are uploaded to the test folder during the submission of the result). This means that you can use the tools you want, but you must add them as data. You can find my ultralytics and pycocotools dataset here. Importantly, the Internet is still present in this notebook to load the model, however, after training it, you will save the model, add it to the data and easily use it for forecasting without the Internet.<br>
</div>

In [29]:
import shutil
import os
import sys
from colorama import Fore

In [30]:
class SetupPipline:
    def __init__(self, display: bool = True):
        self.pycocotools = self.__pycocotools()
        
    @staticmethod
    def __pycocotools() -> str:
        if not os.path.exists("/kaggle/working/packages"):
            shutil.copytree("/kaggle/input/hubmap-tools-ultralytics-and-pycocotools/pycocotools/pycocotools", "/kaggle/working/packages")
            os.chdir("/kaggle/working/packages/pycocotools-2.0.6/")
            os.system("python setup.py install")
            os.system("pip install . --no-index --find-links /kaggle/working/packages/")
            os.chdir("/kaggle/working")
            return "successfully"
    
    def display(self) -> None:
        print(Fore.GREEN+f"\nPycocotools was installed {self.pycocotools}")

In [31]:
pipline = SetupPipline()

running install


/opt/conda/lib/python3.10/site-packages/setuptools/command/install.py:34: SetuptoolsDeprecationWarning: setup.py install is deprecated. Use build and pip and other standards-based tools.
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/setuptools/command/easy_install.py:156: EasyInstallDeprecationWarning: easy_install command is deprecated. Use build and pip and other standards-based tools.
  warnings.warn(


running bdist_egg
running egg_info
writing pycocotools.egg-info/PKG-INFO
writing dependency_links to pycocotools.egg-info/dependency_links.txt
writing requirements to pycocotools.egg-info/requires.txt
writing top-level names to pycocotools.egg-info/top_level.txt
reading manifest file 'pycocotools.egg-info/SOURCES.txt'
reading manifest template 'MANIFEST.in'
writing manifest file 'pycocotools.egg-info/SOURCES.txt'
installing library code to build/bdist.linux-x86_64/egg
running install_lib
running build_py
creating build
creating build/lib.linux-x86_64-3.10
creating build/lib.linux-x86_64-3.10/pycocotools
copying pycocotools/cocoeval.py -> build/lib.linux-x86_64-3.10/pycocotools
copying pycocotools/__init__.py -> build/lib.linux-x86_64-3.10/pycocotools
copying pycocotools/coco.py -> build/lib.linux-x86_64-3.10/pycocotools
copying pycocotools/mask.py -> build/lib.linux-x86_64-3.10/pycocotools
running build_ext
cythoning pycocotools/_mask.pyx to pycocotools/_mask.c


/opt/conda/lib/python3.10/site-packages/Cython/Compiler/Main.py:369: FutureWarning: Cython directive 'language_level' not set, using 2 for now (Py2). This will change in a later release! File: /kaggle/working/packages/pycocotools-2.0.6/pycocotools/_mask.pyx
  tree = Parsing.p_module(s, pxd, full_module_name)


building 'pycocotools._mask' extension
creating build/temp.linux-x86_64-3.10
creating build/temp.linux-x86_64-3.10/common
creating build/temp.linux-x86_64-3.10/pycocotools
gcc -pthread -B /opt/conda/compiler_compat -Wno-unused-result -Wsign-compare -DNDEBUG -fwrapv -O2 -Wall -fPIC -O2 -isystem /opt/conda/include -fPIC -O2 -isystem /opt/conda/include -fPIC -I/opt/conda/lib/python3.10/site-packages/numpy/core/include -I./common -I/opt/conda/include/python3.10 -c ./common/maskApi.c -o build/temp.linux-x86_64-3.10/./common/maskApi.o -Wno-cpp -Wno-unused-function -std=c99


...

gcc -pthread -B /opt/conda/compiler_compat -Wno-unused-result -Wsign-compare -DNDEBUG -fwrapv -O2 -Wall -fPIC -O2 -isystem /opt/conda/include -fPIC -O2 -isystem /opt/conda/include -fPIC -I/opt/conda/lib/python3.10/site-packages/numpy/core/include -I./common -I/opt/conda/include/python3.10 -c pycocotools/_mask.c -o build/temp.linux-x86_64-3.10/pycocotools/_mask.o -Wno-cpp -Wno-unused-function -std=c99


pycocotools/_mask.c: In function ‘__pyx_pf_11pycocotools_5_mask_12iou’:
pycocotools/_mask.c:6140:31: warning: comparison of integer expressions of different signedness: ‘Py_ssize_t’ {aka ‘long int’} and ‘siz’ {aka ‘long unsigned int’} [-Wsign-compare]
 6140 |     if (unlikely(!((__pyx_t_8 == __pyx_v_n) != 0))) {
      |                               ^~
pycocotools/_mask.c:944:43: note: in definition of macro ‘unlikely’
  944 |   #define unlikely(x) __builtin_expect(!!(x), 0)
      |                                           ^


gcc -pthread -B /opt/conda/compiler_compat -shared -Wl,--allow-shlib-undefined -Wl,-rpath,/opt/conda/lib -Wl,-rpath-link,/opt/conda/lib -L/opt/conda/lib -Wl,--allow-shlib-undefined -Wl,-rpath,/opt/conda/lib -Wl,-rpath-link,/opt/conda/lib -L/opt/conda/lib build/temp.linux-x86_64-3.10/./common/maskApi.o build/temp.linux-x86_64-3.10/pycocotools/_mask.o -o build/lib.linux-x86_64-3.10/pycocotools/_mask.cpython-310-x86_64-linux-gnu.so
creating build/bdist.linux-x86_64
creating build/bdist.linux-x86_64/egg
creating build/bdist.linux-x86_64/egg/pycocotools
copying build/lib.linux-x86_64-3.10/pycocotools/cocoeval.py -> build/bdist.linux-x86_64/egg/pycocotools
copying build/lib.linux-x86_64-3.10/pycocotools/__init__.py -> build/bdist.linux-x86_64/egg/pycocotools
copying build/lib.linux-x86_64-3.10/pycocotools/coco.py -> build/bdist.linux-x86_64/egg/pycocotools
copying build/lib.linux-x86_64-3.10/pycocotools/mask.py -> build/bdist.linux-x86_64/egg/pycocotools
copying build/lib.linux-x86_64-3.10/p

zip_safe flag not set; analyzing archive contents...
pycocotools.__pycache__._mask.cpython-310: module references __file__
error: [Errno 2] No such file or directory: '/opt/conda/lib/python3.10/site-packages/numpy-1.24.3.dist-info/METADATA'


Looking in links: /kaggle/working/packages/
Processing /kaggle/working/packages/pycocotools-2.0.6
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for pycocotools: filename=pycocotools-2.0.6-cp310-cp310-linux_x86_64.whl size=93515 sha256=fd9aadee5243db7a0fec141fa13029e3cc9097a8372396baaffdbde0c6565806
  Stored in directory: /root/.cache/pip/wheels/b7/83/32/99474500256e64154dfc568319411b6ff49e96e50f30d9474f
Successfully built pycocotools
  Attempting uninstall: pycocotools
    Found existing installation: pycocotools 2.0.6
    Uninstalling pycocotools-2.0.6:
      Successfully uninstalled pycocotools-2.0.6


In [32]:
pipline.display()


Pycocotools was installed successfully


<div class="alert alert-block alert-info" style="font-size:20px; background-color: #ffc369; font-family:verdana; color: #663500; border: 2px #663500 solid">
    <b>What about submission? 😸</b>
    <br>We have come a long way, done data analysis, trained the model, but what about the submission? We want to send the results, right? Well, I had difficulties with this, and therefore, in order to make life easier for myself, it is possible to help someone, I wrote several classes that allow you to quickly submit without delving into the difficulties that I encountered.<br>
</div>

<div class="alert alert-block alert-info" style="font-size:20px; background-color: #ffc98f; font-family:verdana; color: #b86e1f; border: 2px #b86e1f solid">
    <b>Encode Binary Mask</b>
</div>

In [33]:
import base64
from pycocotools import _mask as coco_mask
import typing as t
import zlib
import pandas as pd
import torchvision.transforms as T
from PIL import Image

In [34]:
class EncodeBinaryMask:
    @staticmethod
    def __checking_mask(mask: np.ndarray) -> np.ndarray:
        if mask.dtype != np.bool:
            raise ValueError(
                "expects a binary mask, received dtype == %s" %
                mask.dtype
            )
        return mask

    @staticmethod
    def __convert_mask(mask: np.ndarray):
        mask_to_encode = mask.astype(np.uint8)
        mask_to_encode = np.asfortranarray(mask_to_encode)
        return mask_to_encode

    @staticmethod
    def __compress_encode(encoded_mask) -> t.Text:
        binary_str = zlib.compress(encoded_mask, zlib.Z_BEST_COMPRESSION)
        base64_str = base64.b64encode(binary_str)
        return base64_str

    def __call__(self, mask: np.ndarray) -> t.Text:
        mask = self.__checking_mask(mask)
        mask_to_encode = self.__convert_mask(mask)
        encoded_mask = coco_mask.encode(mask_to_encode)[0]["counts"]
        base64_str = self.__compress_encode(encoded_mask)
        return base64_str

<div class="alert alert-block alert-info" style="font-size:20px; background-color: #ffc98f; font-family:verdana; color: #b86e1f; border: 2px #b86e1f solid">
    <b>Submission</b>
</div>

In [35]:
class Submission:
    def __init__(self, dirpath: str, model: torch.nn.Module):
        self.__eval_transforms = self.get_transforms()
        self.__model = model
        self.__encoder = EncodeBinaryMask()
        self.__dirpath = dirpath
        self.__filenames = os.listdir(dirpath)
        self.height = 512
        self.width = 512
        
        self.__submission_dict = {
            "id": [],
            "height": [],
            "width": [],
            "prediction_string": []
        }
        
        self.submission = None
    
    @staticmethod
    def get_transforms():
        return T.Compose([
            T.ToTensor(),
            T.Resize(size=(512, 512)),
            T.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.__filenames)

    def __get_columns(self) -> None:
        for filename in self.__filenames:
            path = self.__get_image_path(filename)
            masks = self.__forward(path)
            identifier, height, width, prediction_string = self.__get_cells(filename, masks)
            self.__update_columns(identifier, height, width, prediction_string)

    def __update_columns(self, identifier: str, height: int, width: int, prediction_string: str) -> None:
        self.__submission_dict["id"].append(identifier)
        self.__submission_dict["height"].append(height)
        self.__submission_dict["width"].append(width)
        self.__submission_dict["prediction_string"].append(prediction_string)

    def __get_cells(self, filename: str, masks: list):
        prediction_string = ""
        prediction_string = self.__get_prediction_string(masks, prediction_string)
        identifier = filename.split(".")[0]
        return identifier, self.height, self.width, prediction_string

    def __get_prediction_string(self, masks: list, prediction_string: str) -> str:
        if masks:
            for outputs in masks:
                mask = outputs["mask"]
                mask = np.where(mask > 0.5, 1, 0).astype(np.bool)
                base64_str = self.__encoder(mask)
                confidence = outputs["confidence"]
                prediction_string += f"0 {confidence} {base64_str.decode('utf-8')} "
        else:
            return ""
        return prediction_string

    def __get_image_path(self, filename: str) -> str:
        return os.path.join(
            self.__dirpath, filename
        )

    def __get_image(self, path: str) -> torch.Tensor:
        image = Image.open(path)
        image = np.asarray(image)
        image = self.__eval_transforms(image)
        return image

    def __forward(self, image: torch.tensor) -> list:
        masks = self.__model(image) 
        return masks 

    def submit(self) -> None:
        if not self.submission:
            self.__get_columns()
            self.submission = pd.DataFrame(self.__submission_dict)
            self.submission = self.submission.set_index('id')
            self.submission.to_csv("submission.csv")

In [36]:
__TEST_PATH = "/kaggle/input/hubmap-hacking-the-human-vasculature/test"
sub = Submission(dirpath=__TEST_PATH, model=model)
sub.submit()

/tmp/ipykernel_28/4206064515.py:55: DeprecationWarning: `np.bool` is a deprecated alias for the builtin `bool`. To silence this warning, use `bool` by itself. Doing this will not modify any behavior and is safe. If you specifically wanted the numpy scalar type, use `np.bool_` here.
Deprecated in NumPy 1.20; for more details and guidance: https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations
  mask = np.where(mask > 0.5, 1, 0).astype(np.bool)
/tmp/ipykernel_28/1301857741.py:4: DeprecationWarning: `np.bool` is a deprecated alias for the builtin `bool`. To silence this warning, use `bool` by itself. Doing this will not modify any behavior and is safe. If you specifically wanted the numpy scalar type, use `np.bool_` here.
Deprecated in NumPy 1.20; for more details and guidance: https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations
  if mask.dtype != np.bool:


In [37]:
sub.submission.head()

,height,width,prediction_string
id,,,
72e40acccadf,512,512,0 0.08 eNqVWml34siy/Eu1SgLcfd/MtA1Sbd4NeDcGg9f...


# TO DO
<div class="alert alert-block alert-info" style="font-size:20px; background-color: #ccbda7; font-family:verdana; color: #b86e1f; border: 2px #b86e1f solid">
    <b>Will this competition be over?</b>
    <br>I'm really already quite tired of this competition, but I hope my work is already enough so that they can at least help someone. Why am I saying this? Just so you keep in mind that there are moments that I just didn’t have the strength for and I hope that you can handle them yourself. Namely: use non-max suppression, with lower scores, filter the masks to keep the most likely pixels. I will not return to this notebook, but I would like to know if you have any difficulties.<br>
</div>

<div class="alert alert-block alert-info" style="font-size:20px; background-color: #ccbda7; font-family:verdana; color: #b86e1f; border: 2px #b86e1f solid">
    <b>Thanks a lot for making it to the end 🙃</b>
    <br>I hope you have found this work useful. I will be very grateful if you vote for this work, if it really helped you. Good luck <br>
</div>

![](https://i.pinimg.com/originals/05/01/1c/05011ce8b4b326ed62c70f3eab93f913.gif)